# Synthetic Data Generation for RAG Evaluation

Session 1 built a vector RAG application over a cat health guideline PDF. This
session creates an evaluation dataset for that application and uses the dataset
to compare two retrieval configurations. All generation, embedding, RAG, and
judge requests are routed through Vercel AI Gateway.

The workflow is:

~~~text
corpus -> knowledge graph -> synthetic examples -> human review
       -> LangSmith dataset -> baseline and candidate experiments
~~~

Synthetic examples reduce the cost of getting started, but generated references
are not automatically ground truth. We will inspect and curate them before using
them as evaluation targets.

> This is an educational cat health exercise, not veterinary advice. Generated
> questions and answers must not be used to diagnose, prescribe, or replace a
> veterinarian.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how Ragas builds a knowledge graph for test data generation.
- Distinguish single-hop specific, multi-hop specific, and multi-hop abstract queries.
- Generate and review synthetic questions, reference contexts, and reference answers.
- Route generation, embeddings, RAG, and judge calls through Vercel AI Gateway.
- Upload reviewed examples to a LangSmith dataset.
- Evaluate answer correctness, answer groundedness, and retrieval relevance.
- Run a controlled RAG experiment that changes one variable at a time.

## Table of Contents

- **Breakout Room #1: Synthetic Test Data with Ragas**
  - Task 1: Environment Setup
  - Task 2: Load the Cat Health Corpus
  - Task 3: Build and Enrich a Knowledge Graph
  - Task 4: Inspect the Query Distribution
  - Task 5: Generate and Inspect a Synthetic Test Set
  - Activity #1: Review and Curate the Dataset
- **Breakout Room #2: RAG Evaluation with LangSmith**
  - Task 6: Create a LangSmith Dataset
  - Task 7: Build a Baseline RAG Application
  - Task 8: Define RAG Evaluators
  - Task 9: Run the Baseline Experiment
  - Task 10: Change One Retrieval Variable and Re-Evaluate
  - Activity #2: Compare, Diagnose, and Iterate
  - Advanced Build: Add Robustness and Adversarial Cases

---
# Breakout Room #1
## Synthetic Test Data with Ragas

Ragas uses the source corpus to create a richer representation of its topics and
relationships. Query synthesizers then select scenarios from that representation
and generate questions plus reference answers.

The knowledge graph is a generation aid. It is not the graph used by the RAG
application in Breakout Room #2.

## Task 1: Environment Setup

From the <code>05_Synthetic_Data_Generation_for_RAG_Evals</code> folder:

~~~bash
uv sync
~~~

Then select the environment created by uv as this notebook's kernel.

Required accounts:

- Vercel AI Gateway for generation, embeddings, the RAG answer model, and judges
- LangSmith for the dataset and experiments

A direct OpenAI API key is not required. The OpenAI SDK is used only as a
protocol-compatible client pointed at Vercel AI Gateway.

The default synthetic test set is intentionally small. Ragas generation and
LLM-as-judge evaluation both make multiple model calls, so start small and scale
only after inspecting quality.

### Imports

In [1]:
from __future__ import annotations

import os
from collections import Counter
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import instructor
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, field_validator

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client, evaluate
from openevals.llm import create_llm_as_judge
from openevals.prompts import (
    CORRECTNESS_PROMPT,
    RAG_GROUNDEDNESS_PROMPT,
    RAG_RETRIEVAL_RELEVANCE_PROMPT,
)

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.synthesizers import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    SingleHopSpecificQuerySynthesizer,
    default_query_distribution,
)
from ragas.testset.transforms import (
    CustomNodeFilter,
    SummaryExtractor,
    apply_transforms,
    default_transforms_for_prechunked,
)

/Users/reuben.frith/code/AIE10/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### API Keys, Models, and Cost Controls

The notebook reads model names and budgets from environment variables so you can
tune cost without editing every cell. Vercel AI Gateway exposes an
OpenAI-compatible endpoint, so the OpenAI and LangChain clients only need a
different API key, base URL, and provider-qualified model ID.

See the [Vercel AI Gateway Python documentation](https://vercel.com/docs/ai-gateway/sdks-and-apis/python)
for the current authentication and endpoint details.

LangSmith uses <code>LANGSMITH_TRACING</code>. The older
<code>LANGCHAIN_TRACING_V2</code> name from the source notebook is no longer
needed here.

In [2]:
gateway_api_key = (
    os.environ.get("AI_GATEWAY_API_KEY")
    or os.environ.get("VERCEL_OIDC_TOKEN")
)

if not gateway_api_key:
    gateway_api_key = getpass("Vercel AI Gateway API Key: ")
    os.environ["AI_GATEWAY_API_KEY"] = gateway_api_key

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API Key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault(
    "LANGSMITH_PROJECT",
    "aim-session-5-synthetic-rag-evals",
)

GATEWAY_BASE_URL = os.environ.get(
    "AI_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)
GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)
RAG_MODEL_NAME = os.environ.get(
    "AIM_RAG_MODEL",
    "openai/gpt-5.4-mini",
)
JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)
EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)
TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "6"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "2"))

gateway_models = {
    "generator": GENERATOR_MODEL_NAME,
    "rag": RAG_MODEL_NAME,
    "judge": JUDGE_MODEL_NAME,
    "embedding": EMBEDDING_MODEL_NAME,
}
for role, model_name in gateway_models.items():
    if "/" not in model_name:
        raise ValueError(
            f"{role} model must use a provider-qualified AI Gateway ID: "
            f"{model_name!r}"
        )

print(f"Ragas: {version('ragas')}")
print(f"LangSmith: {version('langsmith')}")
print(f"AI Gateway: {GATEWAY_BASE_URL}")
print(f"Generator model: {GENERATOR_MODEL_NAME}")
print(f"RAG model: {RAG_MODEL_NAME}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Synthetic examples: {TESTSET_SIZE}")
print(f"LangSmith tracing: {os.environ['LANGSMITH_TRACING']}")

Ragas: 0.4.4.dev8+g298b68274
LangSmith: 0.8.16
AI Gateway: https://ai-gateway.vercel.sh/v1
Generator model: openai/gpt-5.4-mini
RAG model: openai/gpt-5.4-mini
Judge model: openai/gpt-5.4-mini
Embedding model: openai/text-embedding-3-small
Synthetic examples: 6
LangSmith tracing: true


## Task 2: Load the Cat Health Corpus

The corpus is the bundled 2021 AAHA/AAFP Feline Life Stage Guidelines PDF used
in Session 1. <code>PyPDFLoader</code> extracts one LangChain document per page,
including page metadata that survives later chunking.

This gives the generator multiple related units to connect:

- hydration and urinary signs
- preventive care and senior care
- dental pain and behavior changes
- monitoring and emergency escalation

In [3]:
corpus_path = Path("data/cat_health_guidelines.pdf")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

pdf_loader = PyPDFLoader(str(corpus_path))
source_documents = pdf_loader.load()
source_documents = [
    document
    for document in source_documents
    if len(document.page_content.strip()) >= 200
]

for index, document in enumerate(source_documents):
    page_number = int(document.metadata.get("page", index)) + 1
    document.metadata.update(
        {
            "source": corpus_path.name,
            "document_type": "feline_life_stage_guidelines",
            "page_number": page_number,
        }
    )

print(f"Loaded {len(source_documents)} text-containing PDF pages")
for document in source_documents[:5]:
    page_number = document.metadata["page_number"]
    print(
        f"- page {page_number}: "
        f"{len(document.page_content)} characters"
    )

Loaded 20 text-containing PDF pages
- page 1: 4913 characters
- page 2: 2084 characters
- page 3: 5955 characters
- page 6: 5673 characters
- page 7: 3588 characters


Inspect one PDF page and its metadata. The metadata is useful for debugging,
trace inspection, and explaining where a retrieved chunk came from.

In [4]:
sample_document = source_documents[0]

print(sample_document.metadata)
print()
print(sample_document.page_content[:800])

{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'document_type': 'feline_life_stage_guidelines', 'page_number': 1}

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177

## Task 3: Build and Enrich a Knowledge Graph

The unrolled workflow makes the generation stages visible:

1. Treat each text-containing PDF page as a pre-chunked Ragas node.
2. Run Ragas extractors, embeddings, and relationship builders.
3. Save the graph so expensive enrichment can be reused.

Ragas remains responsible for graph enrichment and synthetic generation. The
newer pinned Ragas build exposes an official Instructor mode parameter, so its
LLM factory can use AI Gateway tool calls directly without custom wrappers.

In [5]:
gateway_client = OpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=4096,
)
# Provider-qualified Gateway IDs bypass Ragas's GPT-5 parameter detection.
# Keep only the token limit supported by the Gateway route. max_retries is
# consumed locally by Instructor and is not sent to AI Gateway.
generator_llm.model_args = {
    "max_tokens": 4096,
    "max_retries": 3,
}

generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_client,
)

ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

/var/folders/6b/2bnrv4k52z53kz5qmw40mf6r0000gp/T/ipykernel_50039/1199448542.py:21: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = embedding_factory(


Before building the graph, make one small structured-output request through
Ragas. This catches gateway authentication, model availability, and tool-calling
incompatibilities without waiting for every PDF page to retry.

In [6]:
class GatewayToolCallCheck(BaseModel):
    status: str


class NonEmptySummary(BaseModel):
    text: str

    @field_validator("text")
    @classmethod
    def require_text(cls, value):
        value = value.strip()
        if not value:
            raise ValueError("summary text cannot be empty")
        return value


gateway_check = generator_llm.generate(
    "Use the required tool with a short, non-empty status message.",
    GatewayToolCallCheck,
)
if not gateway_check.status.strip():
    raise RuntimeError("AI Gateway returned an empty tool-call check")

print(f"AI Gateway tool-based structured output: {gateway_check.status}")

AI Gateway tool-based structured output: checking


In [7]:
def build_prechunked_knowledge_graph(chunks):
    return KnowledgeGraph(
        nodes=[
            Node(
                type=NodeType.CHUNK,
                properties={
                    "page_content": chunk.page_content,
                    "document_metadata": dict(chunk.metadata),
                },
            )
            for chunk in chunks
            if chunk.page_content.strip()
        ]
    )


generation_chunks = list(source_documents)
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)

print(f"Ragas input chunks: {len(generation_chunks)}")
print(knowledge_graph)

# View the contents for better understanding
print("==================================")
print(knowledge_graph.nodes[1])
print(knowledge_graph.nodes[1].properties["document_metadata"])
print(knowledge_graph.nodes[1].properties["page_content"][:200])

Ragas input chunks: 20
KnowledgeGraph(nodes: 20, relationships: 0)
Node(id: a0a141, type: NodeType.CHUNK, properties: ['page_content', 'document_metadata'])
{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 1, 'page_label': '2', 'document_type': 'feline_life_stage_guidelines', 'page_number': 2}
Introduction
The feline patient ’s life stage is the most fundamental presentation
factor the practitioner encounters in a regular examination visit.
Most of the components of a treatment or healthcar


### Apply Ragas Transforms

Because the PDF loader already gives us coherent page-level chunks, use Ragas'
built-in pre-chunked transform pipeline. It skips headline extraction and
splitting, then applies Ragas summaries, embeddings, themes, named entities,
and relationship builders directly to the PDF pages. The parent-child node
filter is omitted because these page chunks intentionally have no parent nodes.
A non-empty output constraint makes Instructor retry blank Ragas summaries before
the built-in embedding transform runs.

In [8]:
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)
transforms = [
    transform
    for transform in default_transforms_for_prechunked(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )
    if not isinstance(transform, CustomNodeFilter)
]

summary_transform = next(
    transform
    for transform in transforms
    if isinstance(transform, SummaryExtractor)
)
summary_transform.prompt.output_model = NonEmptySummary

print("Ragas transform pipeline:")
for transform in transforms:
    nested = getattr(transform, "transformations", None)
    if nested is None:
        print(f"- {type(transform).__name__}")
    else:
        names = ", ".join(type(item).__name__ for item in nested)
        print(f"- Parallel({names})")

for transform in transforms:
    apply_transforms(
        knowledge_graph,
        transform,
        run_config=ragas_run_config,
    )
    if isinstance(transform, SummaryExtractor):
        empty_summary_nodes = [
            node
            for node in knowledge_graph.nodes
            if not str(node.get_property("summary") or "").strip()
        ]
        if empty_summary_nodes:
            raise RuntimeError(
                "Ragas did not produce non-empty summaries for "
                f"{len(empty_summary_nodes)} PDF chunks"
            )

print(knowledge_graph)

Ragas transform pipeline:
- SummaryExtractor
- Parallel(EmbeddingExtractor, ThemesExtractor, NERExtractor)
- Parallel(CosineSimilarityBuilder, OverlapScoreBuilder)


Applying SummaryExtractor: 100%|██████████| 20/20 [00:36<00:00,  1.84s/it]
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/60 [00:00<?, ?it/s]/Users/reuben.frith/code/AIE10/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/ragas/testset/transforms/base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]: 100%|██████████| 60/60 [01:08<00:00,  1.15s/it]
Applying [CosineSimilarityBuilder, OverlapScoreBuilder]: 100%|██████████| 2/2 [00:00<00:00, 188.98it/s]

KnowledgeGraph(nodes: 20, relationships: 59)


Inspect the graph at a high level. Different Ragas versions may add different
properties, so the notebook avoids depending on one exact internal schema.

In [9]:
node_type_counts = Counter(str(node.type) for node in knowledge_graph.nodes)

print("Node types:")
for node_type, count in node_type_counts.items():
    print(f"- {node_type}: {count}")

print(f"Relationships: {len(knowledge_graph.relationships)}")

for index, node in enumerate(knowledge_graph.nodes[:3], start=1):
    property_names = sorted(node.properties.keys())
    print(f"Node {index} properties: {property_names}")

Node types:
- NodeType.CHUNK: 20
Relationships: 59
Node 1 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 2 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 3 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']


### Save and Reload the Graph

Generated artifacts go in the ignored <code>artifacts/</code> folder so running
the notebook does not add large, machine-generated files to the assignment diff.

In [10]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

knowledge_graph_path = artifacts_dir / "cat_health_knowledge_graph.json"
knowledge_graph.save(str(knowledge_graph_path))

loaded_knowledge_graph = KnowledgeGraph.load(str(knowledge_graph_path))

print(f"Saved graph to {knowledge_graph_path}")
print(loaded_knowledge_graph)
print(loaded_knowledge_graph.nodes[1])

Saved graph to artifacts/cat_health_knowledge_graph.json
KnowledgeGraph(nodes: 20, relationships: 59)
Node(id: 1d75e7, type: NodeType.CHUNK, properties: ['page_content', 'document_metadata', 'summary', 'summary_embedding', 'themes', 'entities'])


#### ❓ Question #1

What information did the Ragas graph transforms add beyond the original text,
and why are the two relationship types important for multi-hop questions?

##### ✅ Answer

We initially have the graph with nodes containing 'page_content' and 'document_metadata'. Then Ragas graph transforms and adds 'entities', 'summary', 'summary_embedding', 'themes'. It also connects the nodes and creates relationships using the embeddings (similar topics using similar language) and extracted entities/themes (common concepts or topics). Multi-hop questions are often more complex and require connecting multiple pieces of information to form an entire picture. By using both embedding based and entity/theme based relationships, Ragas can capture both semantic similarity and explicit conceptual connections, together we can ensure that the multi-hop questions are grounded in chunks that are both topically relevant and conceptually connected, improving accuracy and depth to the generated answers. 

## Task 4: Inspect the Query Distribution

Ragas can synthesize several kinds of questions:

| Query type | What it tests |
|---|---|
| Single-hop specific | Retrieve one concrete fact or recommendation from one context |
| Multi-hop specific | Combine concrete details from multiple related contexts |
| Multi-hop abstract | Connect broader themes or concepts across contexts |

The distribution is part of the evaluation specification. It determines which
behaviors are common in the generated dataset.

In [11]:
query_distribution = default_query_distribution(
    generator_llm,
    kg=loaded_knowledge_graph,
)

print("Available query synthesizers:")
for synthesizer, weight in query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

distribution_total = sum(weight for _, weight in query_distribution)
print(f"Distribution total: {distribution_total:.2f}")

Available query synthesizers:
- single_hop_specific_query_synthesizer: 33%
- multi_hop_abstract_query_synthesizer: 33%
- multi_hop_specific_query_synthesizer: 33%
Distribution total: 1.00


### Define a Custom Distribution

The default is a sensible starting point, but the mix should reflect the
application behavior you care about. This example emphasizes concrete
single-hop questions while preserving coverage of both multi-hop styles.

Adjust the weights below and assign
<code>query_distribution = custom_query_distribution</code> before Task 5 if
you want the generated dataset to use your mix. We define the distribution here
without generating a second test set, which keeps the worked notebook's cost
bounded.

The default helper filters out synthesizers that the enriched graph cannot
support. If a custom multi-hop run reports that no matching relationships exist,
inspect the graph and use only the synthesizers listed by the default distribution.

In [12]:
custom_query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(llm=generator_llm),
        0.50,
    ),
    (
        MultiHopSpecificQuerySynthesizer(llm=generator_llm),
        0.30,
    ),
    (
        MultiHopAbstractQuerySynthesizer(llm=generator_llm),
        0.20,
    ),
]

assert abs(
    sum(weight for _, weight in custom_query_distribution) - 1.0
) < 1e-9

for synthesizer, weight in custom_query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

- single_hop_specific_query_synthesizer: 50%
- multi_hop_specific_query_synthesizer: 30%
- multi_hop_abstract_query_synthesizer: 20%


#### ❓ Question #2

Describe the three query types in your own words. Which type do you expect to be
hardest for a basic dense-retrieval RAG application, and why?

##### ✅ Answer

Single-hop specific: These questions reference a single node and ask for a specific fact. These could be something like "What food should I feed my senior cat?", they feel like the kinds of simple queries you would usually ask google. They are pretty straightforward and should be the easiest questions for basic dense-retrieval to answer where a relevant chunk is retrieved and the answer is likely explicitly stated.

Then there are the 2 multi-hop types, mutli-hop just means the question requires information from multiple nodes to answer, there is likely more complexity and requires a little more connecting of dots to get to an answer. 

Multi-hop specific: These questions are formed by bringing together multiple concrete facts, they are precise. This uses multiple nodes linked by shared entities. An example might be something like "What are the signs of dehydration in my cat and what should i do about it?" This question would have to pull together multiple nodes and focuses on the theme of dehydration. 

Multi-hop abstract: These questions are more open ended and are connected using broader themes. They are not so factual and narrow focused like the multi-hop specific. This uses multiple nodes connected bysemantic similarity/themes. An example might be "What are the common health challenges for senior cats and how  can I support them?". This question is broader and more open ended compared to the multi-hop specific. 

Clearly Multi-hop questions would be harder to answer using a basic dense-retrieval RAG application because they are connecting multiple pieces of information together and not just focusing on a single fact. However the multi-hop abstract would be the hardest this is because it is more open ended. Basic RAG will just take the users query and try to find chunks that have similar semantic scores i.e. are in the same embedding space. Because of the more specific nature of multi-hop specific the embedding created from the query would likely use similar language and terms to the chunks retrieved. In previous lessons we saw that the more specific the question the better the cosine similarity between the chunks and the query. This would give it a greater likelihood of retrieving relevant chunks and answering the question. However the multi hop abstract may not have as highly specific or similar language due to its open endedness which would mean the query that is embedded to find relevant chunks may not be close in embedding space therefore retrieving less relevant chunks and making it harder to answer using a basic dense-retrieval RAG application.

## Task 5: Generate and Inspect a Synthetic Test Set

Each generated row contains:

- <code>user_input</code>: the synthetic question
- <code>reference_contexts</code>: source context selected by the generator
- <code>reference</code>: a generated reference answer
- <code>synthesizer_name</code>: the query strategy that produced the row

The reference is generated from selected source context. It is useful, but it
still needs review for accuracy, clarity, safety, and usefulness.

In [13]:
testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=loaded_knowledge_graph,
)

synthetic_testset = testset_generator.generate(
    testset_size=TESTSET_SIZE,
    query_distribution=query_distribution,
    run_config=ragas_run_config,
)

testset_df = synthetic_testset.to_pandas()

display(
    testset_df[
        [
            "user_input",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Generating Samples: 100%|██████████| 6/6 [00:11<00:00,  1.96s/it]


,user_input,reference,synthesizer_name
0,What is the American Association of Feline Pra...,The 2021 feline life stage guidelines were pre...,single_hop_specific_query_synthesizer
1,Why is a life stage assessment important durin...,The feline patient’s life stage is the most fu...,single_hop_specific_query_synthesizer
2,How do these guidelines use individualized ris...,The guidelines use the cat’s life stage as the...,multi_hop_abstract_query_synthesizer
3,How do feline life stages relate to senior cat...,The feline patient’s life stage is a fundament...,multi_hop_abstract_query_synthesizer
4,According to the American Heartworm Society gu...,Heartworm infection is more difficult to diagn...,multi_hop_specific_query_synthesizer
5,"How does Hazel C. Carney’s work in the ""Hazel ...",In the 2021 AAHA/AAFP Feline Life Stage Guidel...,multi_hop_specific_query_synthesizer


In [14]:
testset_path = artifacts_dir / "cat_health_synthetic_testset.jsonl"
testset_df.to_json(
    testset_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Examples by synthesizer:")
print(testset_df["synthesizer_name"].value_counts())
print()
print(f"Saved candidate examples to {testset_path}")

Examples by synthesizer:
synthesizer_name
single_hop_specific_query_synthesizer    2
multi_hop_abstract_query_synthesizer     2
multi_hop_specific_query_synthesizer     2
Name: count, dtype: int64

Saved candidate examples to artifacts/cat_health_synthetic_testset.jsonl


### Abstracted Ragas Alternative

The graph-first path above makes each Ragas stage inspectable and lets you save
the enriched graph before generation. Ragas also provides a one-call helper for
content that is already chunked:

~~~python
quick_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
quick_testset = quick_generator.generate_with_chunks(
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=transforms,
    run_config=ragas_run_config,
)
~~~

This alternative is shown rather than executed so the notebook does not repeat
the same billable graph enrichment and test-set generation.

#### ❓ Question #3

What are the tradeoffs between the unrolled and one-call Ragas generation paths?
When would you choose each one?

##### ✅ Answer

Straight up we can see that the unrolled path gives us more visibility into each stage as well as the ability to save the enriched graph which is handy if you want to iterate on the generation or use the enriched graph later. We can also iterate on the question mix as we saw the custom distribution example above. With more visibility it is also easier to inspect and debug and tweak stages as we progress. The one call is great if we want to quickly generate test sets and are less worried about inspecting or iterating on the generation. It is great for quick throwaway tests. As data sets get larger and more expensive to generate it may be better to go with the unrolled path to ensure we are generating the kind of data we want at each step before we commit to generating a large data set, this also means we can ensure our token usage is in line with expectations at each stage. It also allows us to cache the enriched graph which can save on costs further down. 

## 🏗️ Activity #1: Review and Curate the Dataset

Review every generated row before uploading it.

For each example, check:

1. Is the question answerable from the reference contexts?
2. Is the reference answer fully supported by those contexts?
3. Is the wording natural for a plausible user?
4. Does the example duplicate another row?
5. Does it preserve the corpus's medical-safety boundaries?

Requirements:

- Remove or repair at least one weak example, if one exists.
- Record why you kept, edited, or removed it.
- Keep the synthesizer name in metadata so you can diagnose query-type failures.

In [15]:
# Activity #1 workspace helper functions

# Instead of displaying via panda dataframe in a table we will print out each example for easier visibility.
def review_row(df, i):
    row = df.iloc[i]
    print("=" * 90)
    print(f"ROW {i}  |  synthesizer: {row['synthesizer_name']}")
    print("=" * 90)
    print(f"\n❓ QUESTION:\n{row['user_input']}\n")
    print("📄 REFERENCE CONTEXTS:")
    ctxs = row['reference_contexts']
    for j, c in enumerate(ctxs if isinstance(ctxs, (list, tuple)) else [ctxs]):
        print(f"  [{j}] {c}\n")
    print(f"✅ REFERENCE ANSWER:\n{row['reference']}\n")

In [ ]:
# Activity #1 workspace

import pandas as pd

# Start with every generated example. Replace this with your reviewed subset.
# We have used 6 generated examples and made a copy that we will review and edit
approved_testset_df = testset_df.copy()
review_status = "review_required"

# View the initial examples prior to review and editing
for i in range(len(testset_df)):
    review_row(testset_df, i)

# If you like the data frame view here it is too
# pd.set_option("display.max_colwidth", 100) # col width is 100 make it None to show full text
# pd.set_option("display.max_rows", 100) # row width is 100 make it None to show all rows     

# display(
#     approved_testset_df[
#         [
#             "user_input",
#             "reference_contexts",
#             "reference",
#             "synthesizer_name",
#         ]
#     ]
# )

ROW 0  |  synthesizer: single_hop_specific_query_synthesizer

❓ QUESTION:
What is the American Association of Feline Practitioners' role in the 2021 feline life stage guidelines?

📄 REFERENCE CONTEXTS:
  [0] VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A noteworthy change from the earlier guidel

### 📝 Activity #1 Notes
I will answer these questions for each row.
1. Is the question answerable from the reference contexts?
2. Is the reference answer fully supported by those contexts?
3. Is the wording natural for a plausible user?
4. Does the example duplicate another row?
5. Does it preserve the corpus's medical-safety boundaries?

Then perform a final decision,
- Decision: Keep, Edit, or Remove
- Reason: Why you kept, edited, or removed it.
- Any safety or grounding issue found: Note any issues found related to safety or grounding.

Review
====================
ROW 0, synthesizer: single_hop_specific_query_synthesizer

1. Yes.
2. Yes.
3. Yes, answer is clear.
4. No duplicates.
5. Yes, no boundaries are crossed.

- Decision: Keep
- Reason: The question is answered, the reference context supports the answer, there is no duplication, the wording is natural, and there are no safety issues.
- Any safety or grounding issue found: None

====================
Row 1, synthesizer: single_hop_specific_query_synthesizer

1. Yes.
2. Yes.
3. Yes.
4. No duplicates.
5. Yes, no boundaries are crossed.

- Decision: Keep
- Reason: No issues found and things look good.
- Any safety or grounding issue found: None

====================
Row 2: synthesizer: multi_hop_abstract_query_synthesizer

1. Yes.
2. Yes.
3. Yes.
4. No duplicates.
5. Yes, no boundaries are crossed.

- Decision: Keep
- Reason: No issues found and things look good.
- Any safety or grounding issue found: None

====================
Row 3: synthesizer: multi_hop_abstract_query_synthesizer

1. Yes.
2. Yes.
3. Yes.
4. No duplicates.
5. Yes, no boundaries are crossed.

- Decision: Keep
- Reason: No issues found and things look good.
- Any safety or grounding issue found: None

====================
Row 4: synthesizer: multi_hop_specific_query_synthesizer

1. Yes.
2. Yes, however <1-hop> pretty much answers the question and the other hops are references. 
3. It is natural but a little confusing cause the question brings together multiple ideas. 
4. No duplicates.
5. Yes, no boundaries are crossed.

- Decision: Edit
- Reason: The question is answerable and supported to a point but the hops degraded in relevance. 
- Any safety or grounding issue found: None.

See repair in code block below.

====================
Row 5: synthesizer: multi_hop_specific_query_synthesizer

1. Partially. The question focuses on Hazel C. Carney and this is not directly answerable. 
2. No. There is no direct link between Carney and the importance of client education with thorough patient histories
3. Partially, question is awkwardly worded making answer unclear.
4. No duplicates.
5. Yes, not a medical safety issue.

- Decision: Remove
- Reason: The question is not answerable from the reference contexts. More research into Hazel C. Carney would be needed to answer the question. 
- Any safety or grounding issue found: None.

See removal in code block below.



In [19]:
# REPAIR row 4
approved_testset_df.loc[4, "user_input"] = (
   "Why is heartworm infection harder to diagnose in cats than in dogs, and does a cat need to be tested for heartworm before starting preventive treatment?"
)
approved_testset_df.loc[4, "reference"] = (
  "Heartworm infection is harder to diagnose in cats than in dogs because cats typically have a lower worm burden, often single-sex infections, and microfilaremia is infrequent. An added complication is HARD (heartworm-associated respiratory disease), an asthma-like inflammatory reaction of the lung tissue to immature larval stages. Because antibody and antigen test results can be difficult to interpret, a thorough understanding of the limitations of both tests is necessary — the American Heartworm Society guidelines provide more detail. Testing does not need to be performed before starting preventive treatment."
)


# Remove row 5 (reset indexes after dropping)
approved_testset_df = approved_testset_df.drop(index=[5]).reset_index(drop=True)

review_status = "student_reviewed"

### 📝 Activity #1 Notes

- Example reviewed:
- Decision:
- Reason:
- Any safety or grounding issue found:

Refer to answers above for each example which was reviewed. 

---
# Breakout Room #2
## RAG Evaluation with LangSmith

We will upload the reviewed examples, build one RAG application, and evaluate two
retrieval settings against the same dataset and judges.

Keeping the dataset and evaluators fixed makes the application change easier to
interpret.

## Task 6: Create a LangSmith Dataset

The dataset stores the question as input and the reviewed synthetic answer plus
reference contexts as outputs. Query type and provenance remain metadata.

A unique suffix prevents accidental duplication when the whole notebook is rerun.
For a long-lived team dataset, use a stable name and manage versions deliberately.

In [20]:
def as_string_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if hasattr(value, "tolist"):
        converted = value.tolist()
        if isinstance(converted, list):
            return [str(item) for item in converted]
    return [str(value)]


if review_status != "student_reviewed":
    raise ValueError(
        "Complete Activity #1, curate approved_testset_df, and set "
        "review_status = 'student_reviewed' before uploading."
    )


langsmith_client = Client()
dataset_name = (
    "aim-session-5-cat-health-synthetic-"
    f"{uuid4().hex[:8]}"
)

langsmith_dataset = langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description=(
        "Ragas-generated questions for the AI Makerspace "
        "cat health RAG lesson."
    ),
    metadata={
        "session": 5,
        "source": "ragas",
        "corpus": str(corpus_path),
    },
)

langsmith_examples = []
for _, row in approved_testset_df.iterrows():
    langsmith_examples.append(
        {
            "inputs": {
                "question": str(row["user_input"]),
            },
            "outputs": {
                "answer": str(row["reference"]),
                "reference_contexts": as_string_list(
                    row["reference_contexts"]
                ),
            },
            "metadata": {
                "synthesizer_name": str(row["synthesizer_name"]),
                "synthetic_reference": True,
                "review_status": review_status,
            },
        }
    )

langsmith_client.create_examples(
    dataset_id=langsmith_dataset.id,
    examples=langsmith_examples,
)

print(f"Created dataset: {dataset_name}")
print(f"Examples uploaded: {len(langsmith_examples)}")

Created dataset: aim-session-5-cat-health-synthetic-8a16fe2c
Examples uploaded: 5


#### ❓ Question #4

Why is it useful to keep <code>synthesizer_name</code>,
<code>synthetic_reference</code>, and review status as metadata instead of
discarding them after upload?

##### ✅ Answer

Keeping <code>synthesizer_name</code>, <code>synthetic_reference</code>, and review status as metadata allows us to slice, trust and audit our eval results. 
- <code>synthesizer_name</code> allows us to slice reults by "Question type", i.e. single_hop_specific, multi_hop_specific, etc. 
- <code>synthetic_reference</code> marks the data as either true for LLm generated or false for not. This allows us to flag the data's provenance where we treat human reviewed and generated answers as a gold standard. 
- <code>review_status</code> allows to capture whether a human sa reviewed the example or not, we would often hold more weight towards human reviewed examples since there has been some level of verification and curation outside of an LLM.

Combining these metadata fields allows us to slice our evaluation and see if there are any trends. It also allows us to have more trust in certain examples, especially those that are human reviewed.

(Note sometimes humans make mistakes too, so a combination of human and LLM review may be best for trusting examples.)


## Task 7: Build a Baseline RAG Application

The baseline uses the same PDF corpus, recursive character chunks, embeddings
and chat generation through Vercel AI Gateway, in-memory Qdrant, and a
context-only answer prompt.

The target returns both the answer and the retrieved contexts. Returning
intermediate retrieval output makes it possible to evaluate retrieval relevance
and answer groundedness without reconstructing hidden steps.

In [21]:
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
rag_documents = rag_splitter.split_documents(source_documents)

rag_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    check_embedding_ctx_length=False,
)
vector_store = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_{uuid4().hex[:8]}",
)

print(f"Source PDF pages: {len(source_documents)}")
print(f"RAG chunks: {len(rag_documents)}")

Source PDF pages: 20
RAG chunks: 255


In [22]:
RAG_SYSTEM_PROMPT = """You are an educational cat health assistant.

Answer the question using only the retrieved context.
If the context is insufficient, say that the corpus does not provide enough
information.

Do not diagnose, prescribe treatment, or present the response as a substitute
for a veterinarian. Clearly preserve any urgent-care guidance found in the
context.

Retrieved context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)
rag_llm = ChatOpenAI(
    model=RAG_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)
answer_chain = rag_prompt | rag_llm | StrOutputParser()

In [24]:
def format_retrieved_document(document) -> str:
    page_number = document.metadata.get("page_number", "unknown")
    source = document.metadata.get("source", "course corpus")
    return (
        f"Page: {page_number}\n"
        f"Source: {source}\n"
        f"{document.page_content}"
    )


def make_rag_target(retrieval_k: int):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_k_{retrieval_k}"
    return target

In [25]:
baseline_retrieval_k = 3
baseline_target = make_rag_target(baseline_retrieval_k)

spot_check_question = (
    "What components should be considered during a feline wellness visit?"
)
baseline_spot_check = baseline_target(
    {"question": spot_check_question}
)

print(baseline_spot_check["answer"])
print()
print("Retrieved contexts:")
for context in baseline_spot_check["contexts"]:
    print("---")
    print(context[:700])

The retrieved guidelines say a feline wellness visit should consider these components:

- Physical and environmental needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

They also note additional important topics such as:

- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Environmental enrichment
- Understanding feline behavior
- Practice team training
- Client education

The corpus does not provide more detail beyond this list.

Retrieved contexts:
---
Page: 1
Source: cat_health_guidelines.pdf
lifelong feline healthcare strategy. The guidelines include a comprehensive table on the components of a feline wellness visit that
provides a framework for systematically implementing an individualized life stage approach to fe line healthcare. Included are
recommendations for managing the most critical health-related factors in relation to a cat’s life stage. These recomm

## Task 8: Define RAG Evaluators

We will evaluate three different relationships:

| Metric | Comparison |
|---|---|
| Answer correctness | Generated answer vs reviewed reference answer |
| Answer groundedness | Generated answer vs contexts retrieved during that run |
| Retrieval relevance | Retrieved contexts vs input question |

These can disagree. A fluent answer can be correct but unsupported by its retrieved
context, or well grounded in context that does not answer the question.

OpenEvals provides reusable prompts, while the small wrapper functions map our
application's dictionary keys into those prompts.

In [26]:
gateway_judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="answer_correctness",
    judge=gateway_judge_llm,
    continuous=True,
)
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT,
    feedback_key="answer_groundedness",
    judge=gateway_judge_llm,
    continuous=True,
)
retrieval_relevance_judge = create_llm_as_judge(
    prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT,
    feedback_key="retrieval_relevance",
    judge=gateway_judge_llm,
    continuous=True,
)

In [27]:
def answer_correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> dict:
    return correctness_judge(
        inputs=inputs["question"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["answer"],
    )


def answer_groundedness(
    outputs: dict,
) -> dict:
    return groundedness_judge(
        context=outputs["contexts"],
        outputs=outputs["answer"],
    )


def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> dict:
    return retrieval_relevance_judge(
        inputs=inputs["question"],
        context=outputs["contexts"],
    )


rag_evaluators = [
    answer_correctness,
    answer_groundedness,
    retrieval_relevance,
]

#### ❓ Question #5

Give one example where answer correctness and groundedness could disagree. What
would that disagreement tell you to inspect in the trace?

##### ✅ Answer

An example like this could be where the llm is able to generate a correct answer but it is not grounded in the retrieved context. This could be because the LLM is able to answer using its own knowledge and not rely on the retrieved context. An example could be a question like "What are the signs a cat is misbehaving?", if the retrieval corpus references cat health and doesn't touch on behaviour, the LLM will likely not retrieve a relevant chunk. However if the LLM has previous knowledge about cat behaviour and is able to answer then the would be a case of a correct answer but not grounded. It is indicating that the LLM has bypassed the retrieval and using its own knowledge to answer. 
The disagreement could indicate a few things,
1. The retrieved chunks are not relevant to the question, so the LLM is relying on its own knowledge to answer. This would suggest an issue with retrieval relevance. There may be an issue with the retrieval configuration, eg chunk sizing, over lapping,embedding model, or any other part of the retrieval pipeline.
2. The prompt may not be effectively instructing the LLM to ground its answers and only refer to the retrieved context. Doing this may not always be desirable, but if grounding is important then this would be an issue, especially in regulated industries like medicine, law or finance. 
3. There may be an issue with the LLM's ability to interpret the retrieved context correctly, which could indicate a need for better prompt engineering or model fine-tuning.
4. Or lastly making sure the judge did not make a mistake when evaluating the answer's correctness or groundedness.

## Task 9: Run the Baseline Experiment

LangSmith runs the target once for each dataset example, applies all evaluators,
and groups the traces under one experiment.

After the run, open the experiment URL and inspect individual failures. Aggregate
scores alone do not explain whether the problem came from the generated dataset,
retrieval, prompting, or the judge.

In [29]:
baseline_results = evaluate(
    baseline_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-baseline-k3",
    description=(
        "Baseline cat health RAG with 500-character chunks "
        "and retrieval k=3."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Baseline experiment: {baseline_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-baseline-k3-3dcaf49c' at:
https://smith.langchain.com/o/711d42aa-eb36-4dde-a5c5-0301b6a55fbc/datasets/5aff6a2e-5b99-4ae0-a29c-5dae58fced11/compare?selectedSessions=f5477d14-4c34-4cbb-b376-15650f51c864




5it [00:21,  4.21s/it]

Baseline experiment: cat-health-rag-baseline-k3-3dcaf49c


### Baseline Inspection Notes

Add cat-health-rag-baseline-k3-3dcaf49c.csv for reference.

- Lowest-scoring example: The Lowest scoring example was row 5 for answer_correctness at 0.45.
- Metric that failed: Answer correctness did poorly with an average of 0.78 with row 5 scoring the lowest at 0.45.
- Was the synthetic reference valid? 
We can tell this by looking at all 3 metrics, a poor reference would likely have some disagreement between them.
The two examples that have the most disagreement is Row 5 (i.e. our previous Row 4 which we edited). The answer groundedness is 0.82 while the retrieval relevance is 0.65 and the answer correctness is 0.45. This indicates the retrieved context was somewhat relevant to the question and the answer was somewhat grounded in that context, but the answer was not correct. Inspecting the reference answer we see that he LLM stated "However, the corpus does not provide enough information to explain in detail". This is good model behaviour and explains the low correctness score. Also this was the row where we edited the question and answer but did not change the retrieved context. This shows the importance of ensuring that the retrieved context is relevant to the question and answer, if it is not then the LLM may struggle to generate a correct answer.
- Was the retrieved context relevant and sufficient? Retrieved relevance was 0.9 on average but row 5 and a score of 0.65, this shows that the for most questions the retrieved context was relevant but for row 5 it was less relevant. This could be a factor in the low answer correctness score for row 5, if the retrieved context is not relevant enough to answer the question then the LLM may struggle to generate a correct answer. This directly aligns with out effort where we update the question and answer but did not update the retrieved context. 
- Did the answer add unsupported information?
The answer_groundedness was on average 0.95, with row 5 scoring 0.82. Looking at the chunks collected they are somewhat relevant and likely have some useful/relevant info. However when we see the score of 0.45 for answer correctness we see that the LLM stated "However, the corpus does not provide enough information to explain in detail". This is good model behaviour, it is not adding unsupported information and is acknowledging its limitations.

## Task 10: Change One Retrieval Variable and Re-Evaluate

The source notebook changed chunk size, embedding model, retriever settings, and
prompt style at the same time. That makes any score change hard to explain.

Here we change only retrieval depth:

~~~text
baseline:  k = 3
candidate: k = 6
~~~

The chunks, embeddings, vector store, answer model, prompt, dataset, and evaluators
remain fixed.

In [30]:
candidate_retrieval_k = 6
candidate_target = make_rag_target(candidate_retrieval_k)

candidate_spot_check = candidate_target(
    {"question": spot_check_question}
)

print(candidate_spot_check["answer"])
print()
print(
    "Retrieved context count:",
    len(candidate_spot_check["contexts"]),
)

The corpus says a feline wellness visit should consider these components:

- Food and environmental needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also notes other important topics such as:

- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Environmental enrichment
- Understanding feline behavior
- Practice team training
- Client education

The guidelines describe this as part of an individualized, life stage approach to feline healthcare.

Retrieved context count: 6


In [31]:
candidate_results = evaluate(
    candidate_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate-k6",
    description=(
        "Candidate cat health RAG with the same index and "
        "retrieval k increased from 3 to 6."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": candidate_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Candidate experiment: {candidate_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-candidate-k6-399e679b' at:
https://smith.langchain.com/o/711d42aa-eb36-4dde-a5c5-0301b6a55fbc/datasets/5aff6a2e-5b99-4ae0-a29c-5dae58fced11/compare?selectedSessions=e02c0b0f-c7c9-45fa-acc6-e91d9b72fe55




5it [00:21,  4.25s/it]

Candidate experiment: cat-health-rag-candidate-k6-399e679b


#### ❓ Question #6

Why is changing one variable at a time useful? If correctness improves while
retrieval relevance falls, what might the larger value of <code>k</code> be doing?

##### ✅ Answer

Changing one variable at a time is best practice and allows us to attribute any change in our metrics to the change in the one variable. This is standard practice in scientific experiments where you want to isolate cause and effect. 
Increasing k means we are retrieving more chunks. If retrieval relevance falls while correctness improves it means that the additional chunks we retrieved are helping the LLM generate more correct answer, but the chunks are slightly less relevant to the question. That is the additional chunks may be slightly "diluting" the relevancy of the retrieved context by providing more information that is not closely related to the question, but it still helps answer it better. 

## 🏗️ Activity #2: Compare, Diagnose, and Iterate

Compare the baseline and candidate experiments in LangSmith.

Requirements:

1. Record the mean score for each evaluator in both experiments.
2. Inspect at least two examples whose scores changed.
3. Decide whether <code>k=6</code> improved the application overall.
4. Choose one new variable to test: chunk size, chunk overlap, embedding model,
   prompt, or retrieval depth.
5. State your prediction before running the experiment.
6. Run a third experiment and explain the result.

Keep the reviewed dataset and evaluators fixed. If you discover that an example
itself is invalid, fix or remove the example and treat that as dataset maintenance,
not an application improvement.

### 📝 Activity #2 Notes

The below values can be seen in the attached CSVs in the current folder.

### Mean Scores for k=3 vs k=6
| Evaluator | k=3 Mean Score | k=6 Mean Score | Change |
|---|---|---|---|
| Answer Correctness | 0.78 | 0.86 | +0.08 |
| Answer Groundedness | 0.95 | 0.98 | +0.03 |
| Retrieval Relevance | 0.9 | 0.95 | +0.05 |

Overall we can see an improvement scores by increasing k fro 3 to 6. There was only a slight decrease in latency (P50) from 2.52 seconds to 2.46 seconds, however this is not significant and a metric we are not focused on optimizing at this point in time. Additionally there was increase in tokens used by going from k=3 to k=6 across the board for all examples which is expected since we are retrieving more chunks which is consuming more context and likely leading to longer answers.

### Inspections
Example 1: Row 5 (multi_hop_abstract_query_synthesizer)
| Evaluator | k=3 Score | k=6 Score | Change |
|---|---|---|---|
| Answer Correctness | 0.45 | 0.55 | +0.10 |
| Answer Groundedness | 0.82 | 0.98 | +0.16 |
| Retrieval Relevance | 0.65 | 0.86 | +0.21 |


Example 2: Row 4 (multi_hop_abstract_query_synthesizer)
| Evaluator | k=3 Score | k=6 Score | Change |
|---|---|---|---|
| Answer Correctness | 0.7 | 1.0 | +0.30 |
| Answer Groundedness | 0.98 | 1.0 | +0.02 |
| Retrieval Relevance | 0.93 | 0.88 | -0.05 |

In both the examples we can see an improvement in answer correctness and groundedness however increasing k from 3 to 6 for Row 4 did decrease the retrieval relevance but improved it for Row 5. We discussed the reason for seeing the drop in retrieval relevance before where more chunks are likely diluting the overall relevancy of the retrieved context but still providing some information that is helping the LLM generate a more correct answer. However in Row 5 adding more chunks actually improved the retrieval relevance. Inspecting Row 5 further we can actually see that the LLM answered the question better and did not answer with "However, the corpus does not provide enough information to explain in detail:" this is great sign and shows that the additional chunks retrieved help provide more relevant information.

Setting k=6 has improved the overall performance for our application as we focus on improving answer correctness, groundedness and retrieval relevance. It is a setting worth keeping as we continue to iterate and improve our application.

### New Baseline
Because there is an improvement in correctness, groundedness and retrieval relevance, I would choose to keep k=6 as the new baseline and continue iterating from there.

### New Variable to Test: Increase Chunk Size (500 -> 1000)
I will test the impact of increasing chunk size. Personally I feel that increasing chunk size will have a similar impact to increasing k, where we will see an increase in correctness and groundness whilst retrieval relevance may drop slightly/ remain steady. Increasing chunk size means we are giving the LLM more context in each chunk which may help it generate a more correct answer, since each chunk will be more comprehensive and have more information. However with this the "relevancy" of each chunk may drop slightly since we are including more information in each chunk, some of which may not be directly relevant to the question.

In [33]:
# Activity #2 workspace - Experiment #3: increase chunk size (500 -> 1000)
#
# Controlled change: ONLY chunk_size changes. Overlap (75), embedding model,
# answer model, prompt, retrieval depth (k=6), dataset, and evaluators are all
# held fixed, so any score movement is attributable to chunk size alone.
# We hold k=6 because the k=6 candidate was adopted as the new default, so the
# clean comparison is chunk1000-k6 vs candidate-k6.

large_chunk_size = 1000
large_chunk_overlap = 75  # kept fixed at the baseline value

large_chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=large_chunk_size,
    chunk_overlap=large_chunk_overlap,
)
large_chunk_documents = large_chunk_splitter.split_documents(source_documents)

large_chunk_vector_store = QdrantVectorStore.from_documents(
    documents=large_chunk_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_chunk{large_chunk_size}_{uuid4().hex[:8]}",
)

print(
    f"RAG chunks at size {large_chunk_size}: {len(large_chunk_documents)} "
    f"(500-char baseline produced {len(rag_documents)} chunks)"
)


def make_rag_target_for_store(store, retrieval_k):
    """Same output contract as make_rag_target, but for an arbitrary store."""
    retriever = store.as_retriever(search_kwargs={"k": retrieval_k})

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_chunk{large_chunk_size}_k{retrieval_k}"
    return target


# Hold k fixed at the adopted k=6 so chunk_size is the only changed variable.
large_chunk_retrieval_k = candidate_retrieval_k
large_chunk_target = make_rag_target_for_store(
    large_chunk_vector_store,
    large_chunk_retrieval_k,
)

large_chunk_spot_check = large_chunk_target(
    {"question": spot_check_question}
)
print(large_chunk_spot_check["answer"])
print()
print(
    "Retrieved context count:",
    len(large_chunk_spot_check["contexts"]),
)

RAG chunks at size 1000: 120 (500-char baseline produced 255 chunks)
The corpus says a feline wellness visit should use a framework that includes:

- behavior and environmental needs  
- elimination  
- life stage nutrition and weight management  
- oral health  
- parasite control  
- vaccination  
- zoonoses and human safety  
- recommended diagnostics based on life stage

It also notes that the full guidelines provide a comprehensive table for systematically implementing an individualized, life-stage approach to feline healthcare.

Retrieved context count: 6


In [34]:
large_chunk_results = evaluate(
    large_chunk_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-chunk1000-k6",
    description=(
        "Chunk-size experiment: 1000-character chunks (up from 500) "
        "with retrieval k=6 held fixed; all else identical to candidate-k6."
    ),
    metadata={
        "chunk_size": large_chunk_size,
        "chunk_overlap": large_chunk_overlap,
        "retrieval_k": large_chunk_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "chunk_size",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Chunk-size experiment: {large_chunk_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-chunk1000-k6-4feea007' at:
https://smith.langchain.com/o/711d42aa-eb36-4dde-a5c5-0301b6a55fbc/datasets/5aff6a2e-5b99-4ae0-a29c-5dae58fced11/compare?selectedSessions=fdf00867-f2b2-4c32-bb8c-6cc41bcf52e2




5it [00:20,  4.02s/it]

Chunk-size experiment: cat-health-rag-chunk1000-k6-4feea007


### Result from Increasing Chunk Size

#### Mean Scores for k=6, chunk size 500 vs chunk size 1000
| Evaluator | Chunk Size 500 Mean Score (k=6) | Chunk Size 1000 Mean Score (k=6)| Change |
|---|---|---|---|
| Answer Correctness | 0.86 | 0.81 | -0.05 |
| Answer Groundedness | 0.98 | 0.91 | -0.07 |
| Retrieval Relevance | 0.95 | 0.97 | +0.02 |

Overall we can see that increasing the chunk size from 500 to 1000 actually decreased the answer correctness and groundedness while slightly improving retrieval relevance. This is interesting and not what I expected. I expected that increasing chunk size would provide more context in each chunk which would help the LLM generate a more correct answer, however it seems that the larger chunk size may be providing too much information in each chunk which is making it harder for the LLM to identify the most relevant information to answer the question, this could be leading to a decrease in correctness and groundedness. The slight increase in retrieval relevance could be because with larger chunks we are more likely to retrieve chunks that contain some relevant information, however the additional information in those chunks may be making it harder for the LLM to generate a correct and well grounded answer.

### Inspections

Row 5 (multi_hop_abstract_query_synthesizer)
| Evaluator | Chunk Size 500 Score (k=6) | Chunk Size 1000 Score (k=6) | Change |
|---|---|---|---|
| Answer Correctness | 0.55 | 0.90 | +0.35 | 
| Answer Groundedness | 0.98 | 0.98 | 0.00 |
| Retrieval Relevance | 0.86 | 0.98 | +0.12 |


Row 1 (single_hop_specific_query_synthesizer)
| Evaluator | Chunk Size 500 Score (k=6) | Chunk Size 1000 Score (k=6) | Change |
|---|---|---|---|
| Answer Correctness | 0.98 | 0.6 | -0.38 |
| Answer Groundedness | 0.94 | 0.75 | -0.19 |
| Retrieval Relevance | 1.00 | 0.90 | -0.10 |

Inspecting these two rows paints an interesting picture 
we can see that for Row 5 increasing the chunk size actually improved the answer correctness and retrieval relevance while keeping the groundedness the same. This is interesting because it goes against the overall trend we saw where increasing chunk size decreased correctness and groundedness. It seems that for this specific example, having more information in each chunk actually helped the LLM generate a more correct answer and retrieve more relevant chunks. This could be because the question in Row 5 is a multi-hop abstract question which may benefit from having more context in each chunk to help connect the dots and generate a correct answer.
While for Row 1 increasing the chunk size decreased the answer correctness, groundedness and retrieval relevance. It seems that for this specific example, having more information in each chunk actually made it harder for the LLM to identify the most relevant information to answer the question which led to a decrease in correctness, groundedness and retrieval relevance. This could be because the question in Row 1 is a single-hop specific question which may not benefit from having more context in each chunk since it is looking for a specific fact and having too much information in each chunk may make it harder for the LLM to identify that specific fact.

Additionally we can see that the latency (P50) decreased from 2.46 seconds to 2.25 seconds when we increased the chunk size from 500 to 1000, latency at this scale and using this is highly dependent on infrastructure that is outside of our control. Since the change in latency is not changing our user experience in a meaningful way it is ok. Evidently the tokens used increased when we increased the chunk size, this is expected since larger chunks means we are consuming more context which leads to more tokens being used.

### Conclusion

In conclusion, while increasing the chunk size from 500 to 1000 provided more context in each chunk it actually decreased the overall performance in our application in terms of answer correctness and groundedness, while slightly improving retrieval relevance. This suggests that there may be an optimal chunk size that provides enough context without overwhelming the LLM with too much information, and that this optimal chunk size may vary depending on the type of question being asked.

Also because we only increased the chunk size and kept all other variables the same we can be confident that the changes in our metrics are due to the change in chunk size, this is why it is important to change one variable at a time when running experiments. I think it is important to understand what we are optimising then focus on the variables that would most likely impact that metric. Also you can clearly see that my prediction was wrong, so it is important to run these experiments and not just assume.

The way we design our RAG application is highly dependent on the use case and the type of questions we expect to receive. For example if we expect to receive more multi-hop abstract questions then it may be beneficial to have a larger chunk size to provide more context in each chunk, while if we expect to receive more single-hop specific questions then it may be beneficial to have a smaller chunk size to make it easier for the LLM to identify specific facts. It is important to experiment with different configurations and evaluate their performance using a dataset and evaluators that are relevant to our use case.

## Advanced Build: Add Robustness and Adversarial Cases

Synthetic data can cover failure modes as well as happy-path questions.

Add at least three reviewed cases such as:

- A user asks for a diagnosis or medication dose that the corpus cannot support.
- A prompt-injection attempt asks the assistant to ignore its context-only policy.
- An unrelated question should trigger an insufficient-context response.
- Retrieved text contains a malicious instruction that should be treated as data,
  not as an instruction.

For each case, define the expected behavior and an evaluator that measures it.
Track normal-task performance and attack resistance separately so a system does
not appear safe merely because it refuses everything.

## Final Takeaways

- Synthetic data is a starting point for evaluation, not a replacement for human
  review or production examples.
- The knowledge graph and query distribution shape which capabilities the dataset
  measures.
- Store provenance and review metadata so failures can be traced back to the data.
- Return retrieval output from the target when retrieval and grounding matter.
- Evaluate retrieval, grounding, and answer quality separately.
- Change one application variable at a time when you want an interpretable result.